In [0]:
# DBTITLE 1,Importação de Funções
from pyspark.sql.functions import (
    col, to_timestamp, datediff, round, current_timestamp, when, lower, trim
)

CATALOG = "workspace"
SCHEMA = "default"

# DBTITLE 2,1. Silver Orders (Limpeza e Cálculo de Prazos)
df_orders = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")

df_silver_orders = (
    df_orders
    .filter(col("order_status") == "delivered")  # Apenas pedidos entregues para análise de performance
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at")))
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date")))
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))
    # Métrica de negócio: tempo de entrega em dias
    .withColumn("delivery_time_days", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    # Indicador se a entrega foi em atraso (1 = Sim, 0 = Não)
    .withColumn("is_delayed", when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1).otherwise(0))
    .withColumn("_updated_at", current_timestamp())
)

df_silver_orders.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")
print("✓ Tabela silver_orders criada com sucesso!")

# DBTITLE 3,2. Silver Customers (Tratamento de Localização)
df_customers = spark.table(f"{CATALOG}.{SCHEMA}.bronze_customers")

df_silver_customers = (
    df_customers
    .withColumn("customer_city", lower(trim(col("customer_city"))))
    .withColumn("customer_state", trim(col("customer_state")))
    .withColumn("_updated_at", current_timestamp())
)

df_silver_customers.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_customers")
print("✓ Tabela silver_customers criada com sucesso!")

# DBTITLE 4,3. Silver Products (Tratamento de Categorias)
df_products = spark.table(f"{CATALOG}.{SCHEMA}.bronze_products")

df_silver_products = (
    df_products
    # Preencher categorias nulas com 'outros'
    .fillna({"product_category_name": "nao_definido"})
    .withColumn("product_category_name", lower(trim(col("product_category_name"))))
    .withColumn("_updated_at", current_timestamp())
)

df_silver_products.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_products")
print("✓ Tabela silver_products criada com sucesso!")

# DBTITLE 5,4. Silver Order Items & Payments
df_items = spark.table(f"{CATALOG}.{SCHEMA}.bronze_order_items")
df_silver_items = df_items.withColumn("_updated_at", current_timestamp())
df_silver_items.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_order_items")
print("✓ Tabela silver_order_items criada com sucesso!")

df_payments = spark.table(f"{CATALOG}.{SCHEMA}.bronze_payments")
df_silver_payments = df_payments.withColumn("_updated_at", current_timestamp())
df_silver_payments.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_payments")
print("✓ Tabela silver_payments criada com sucesso!")

print("\n--- Processamento da Camada Prata Concluído! ---")

✓ Tabela silver_orders criada com sucesso!
✓ Tabela silver_customers criada com sucesso!
✓ Tabela silver_products criada com sucesso!
✓ Tabela silver_order_items criada com sucesso!
✓ Tabela silver_payments criada com sucesso!

--- Processamento da Camada Prata Concluído! ---
